In [ ]:
from run_pol_apr_questions import run_all
df = run_all()

In [ ]:
from dex_credis_simulation import CredisSimulation
from f_firerate_calibration import FirerateCalibration

sim = CredisSimulation(
    coen_price_path='/Users/aminakaltayeva/Desktop/crypto_joyslab/misc/coen_price_simulation_v6.xlsx',
    s_curve_path='/Users/aminakaltayeva/Desktop/crypto_joyslab/misc/s_curve_price_simulation_v2.xlsx',
)
sim.simulation_length = 500
sim.firerate_tvl_mcap_fraction = 0.1
# sim.firerate_tvl_mcap_fraction = 
# sim.firerate_compute_every_n_days = 15
sim.run()
df = sim.results

cal = FirerateCalibration(sim.results)
ulf = cal.user_loss_fairness(verbose=False)
dly = cal.conversion_delay(verbose=False)

print(f"     Потери в стресс-дни: средн {ulf.panic_user_mean_loss_pct:.1f}%, "
        f"P95 {ulf.panic_user_p95_loss_pct:.1f}%; "
        f"в норме {ulf.normal_user_mean_loss_pct:.1f}%")
print(f"     Ожидание конвертации: медиана "
        f"{dly['wait_median']:.1f} дн (норма {dly['wait_median_calm']:.1f}, "
        f"стресс {dly['wait_median_stress']:.1f}), "
        f"P95 {dly['wait_p95']:.1f} дн")

In [ ]:
df[['FirerateStressLevel', 'FirerateEffectiveRate',
       'FirerateStress1Hours', 'FirerateStress2Hours', 'SpreadEnd',
       'SpreadEndNoFR',]]['FirerateStressLevel']

In [ ]:
from dex_credis_simulation import CredisSimulation
from f_firerate_calibration import FirerateCalibration

sim = CredisSimulation(
    coen_price_path='/Users/aminakaltayeva/Desktop/crypto_joyslab/misc/coen_price_simulation_v6.xlsx',
    s_curve_path='/Users/aminakaltayeva/Desktop/crypto_joyslab/misc/s_curve_price_simulation_v2.xlsx',
)
sim.simulation_length = 1000
sim.firerate_tvl_mcap_fraction = 0.2
# sim.firerate_tvl_mcap_fraction = 
# sim.firerate_compute_every_n_days = 15
sim.run(verbose=False)
df = sim.results

cal = FirerateCalibration(sim.results)
ulf = cal.user_loss_fairness(verbose=False)
dly = cal.conversion_delay(verbose=False)

print(f"     Потери в стресс-дни: средн {ulf.panic_user_mean_loss_pct:.1f}%, "
        f"P95 {ulf.panic_user_p95_loss_pct:.1f}%; "
        f"в норме {ulf.normal_user_mean_loss_pct:.1f}%")
print(f"     Ожидание конвертации: медиана "
        f"{dly['wait_median']:.1f} дн (норма {dly['wait_median_calm']:.1f}, "
        f"стресс {dly['wait_median_stress']:.1f}), "
        f"P95 {dly['wait_p95']:.1f} дн")
print()

In [ ]:
sim.gratis_balance.mean()

In [ ]:
df[['GratisRequested', 'GratisConverted', 'ConversionQueueGratis']]

In [ ]:
df[['GratisRequested', 'GratisConverted', 'ConversionQueueGratis']].plot()

In [ ]:
df['TVLRequiredNoFR'].value_counts()

In [ ]:
df[['FirerateStressLevel', 'FirerateEffectiveRate',
       'FirerateStress1Hours', 'FirerateStress2Hours', 'SpreadEnd',
       'SpreadEndNoFR', 'GratisConvertedNoFR', 'CoenFromConversionNoFR',
       'DepthEnd', 'DHatEnd', 'LPHealth', 'LPHealthNoFR',
       'ConversionQueueGratis', 'PoolTVL', 'PoolTVLPrivate', 'PoolAPRRealized',
       'PoolVolumeDay', 'PoolTVLNoFR', 'PoolTVLPrivateNoFR',
       'PoolAPRRealizedNoFR', 'TVLRequired', 'PolMinUSD', 'PolMinFrac',
       'GapUSD', 'ImpliedAPRGap']]

In [ ]:
from dex_credis_simulation import CredisSimulation
import pandas as pd

df = pd.DataFrame()

for pol, apr in [(100_000, 0.10), (500_000, 0.25), (2_000_000, 0.25)]:
    sim = CredisSimulation(
        coen_price_path='/Users/aminakaltayeva/Desktop/crypto_joyslab/misc/coen_price_simulation_v6.xlsx',
        s_curve_path='/Users/aminakaltayeva/Desktop/crypto_joyslab/misc/s_curve_price_simulation_v2.xlsx',
    )
    sim.firerate_compute_every_n_days = 15
    sim.run_pol_apr(pol, apr, fee_tier=0.01, verbose=True)
    temp_df = sim.results
    temp_df['pol'] = pol

    df = pd.concat([df, temp_df])

In [ ]:
from f_firerate_calibration import FirerateCalibration
cal = FirerateCalibration(sim.results)
cal.run_all()

In [ ]:
from dex_market import print_attack_report
print_attack_report(sim.pool_fr, sim.dex_proto,
                    daily_volume_usd=sim.results['CoenFromConversion'].tail(30).mean()
                                     * sim.results['CoenPrice'].iloc[-1])

In [ ]:
from firerate_hourly_dex import probe_endogenous_dex, find_min_panic_threshold

# выдержит ли текущее состояние bank-run-шок
r = probe_endogenous_dex(sim.gratis_balance, sim.joined_customer_base,
                         sim.pool_fr, clf_level=0, proto=sim.dex_proto,
                         hours=72, panic_sens=4.0,
                         initial_displacement=0.10, initial_lp_shock=0.3, seed=42)
print(r.spiral, r.hours_severe, r.min_d_hat, r.final_lp_health)

# margin of safety: порог паники с Firerate и без
thr_on  = find_min_panic_threshold(sim.gratis_balance, sim.joined_customer_base,
                                   sim.pool_fr, sim.dex_proto, firerate_on=True,  seed=42)
thr_off = find_min_panic_threshold(sim.gratis_balance, sim.joined_customer_base,
                                   sim.pool_fr, sim.dex_proto, firerate_on=False, seed=42)

In [ ]:
from lp_resilience import lp_shock_frontier, min_pol_requirement, plot_lp_frontier

fr = lp_shock_frontier(sim.gratis_balance, sim.joined_customer_base,
                       sim.pool_fr, sim.dex_proto, firerate_on=True)
lp_shock_frontier(sim.gratis_balance, sim.joined_customer_base,sim.pool_fr, sim.dex_proto, firerate_on=False)   # хрупкость рынка без Firerate
pol = min_pol_requirement(sim.gratis_balance, sim.joined_customer_base,
                          sim.pool_fr, sim.dex_proto)
